# Component Summation

Aggregate projections from the six per-component notebooks into total GMSL.

**Components:** Thermosteric (ocean), Glaciers, Greenland, EAIS, Antarctic Peninsula, WAIS

**Method:** Load MC sample ensembles from `component_results.h5`, sum sample-by-sample to preserve correlations within each draw, then compute percentiles on the total.

**Sections:**
0. Imports & configuration
1. Load all component projections
2. Sum components → total GMSL
3. Summary table (2050, 2100, 2150)
4. Projection fan plots
5. Component stack & variance decomposition
6. IPCC AR6 comparison

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.style.use('seaborn-v0_8-poster')

sys.path.insert(0, '.')
from component_io import (
    load_all_projections, load_component, list_components,
    PROJ_SSPS, PROJ_YEARS, DEFAULT_H5_PATH,
)
from component_projections import (
    read_ipcc_component_nc, ipcc_extract,
    get_our_stats, get_ipcc_stats, stats_dict,
)
from component_analysis import compute_variance_fractions
from component_plotting import (
    SSP_COLORS, COMP_COLORS, M_TO_MM,
    plot_component_projections,
    plot_variance_decomposition,
)

RAW_DIR = '../data/raw'
FIG_DIR = '../figures'
CONF_BASE = f'{RAW_DIR}/ipcc_ar6/slr/ar6/global/confidence_output_files'

# Display names for components (HDF5 key → label)
COMP_LABELS = {
    'ocean': 'Thermosteric',
    'glacier': 'Glaciers',
    'greenland': 'Greenland',
    'eais': 'EAIS',
    'apeninsula': 'Peninsula',
    'wais': 'WAIS',
}

# SSP name → IPCC code
SSP_TO_CODE = {
    'SSP1-2.6': 'ssp126', 'SSP2-4.5': 'ssp245',
    'SSP3-7.0': 'ssp370', 'SSP5-8.5': 'ssp585',
}

## 1. Load All Component Projections

In [ ]:
# Inventory
list_components()

In [ ]:
# Load projection ensembles for all components
proj_years, all_proj = load_all_projections()

available = sorted(all_proj.keys())
print(f'Loaded components: {available}')
print(f'Projection years: {proj_years[0]:.0f}–{proj_years[-1]:.0f} ({len(proj_years)} pts)')

# Check SSP coverage per component
for comp in available:
    ssps = sorted(all_proj[comp].keys())
    has_samples = all('samples' in all_proj[comp][s] for s in ssps)
    print(f'  {comp:12s}  SSPs: {", ".join(ssps)}  samples: {has_samples}')

## 2. Sum Components → Total GMSL

Sum sample-by-sample across components to propagate uncertainty correctly. Each MC draw $i$ gives one total trajectory:

$$H_{\mathrm{total}}^{(i)}(t) = \sum_{c} H_c^{(i)}(t)$$

Percentiles are then computed across the ensemble of total trajectories.

In [ ]:
# ── Build comp_projections in the {ssp: {Label: dict}} layout ──
# This is the format expected by plot_component_projections and
# compute_variance_fractions.
comp_projections = {}

for ssp in PROJ_SSPS:
    comp_projections[ssp] = {}

    # Per-component entries (relabelled)
    for hdf_key, label in COMP_LABELS.items():
        if hdf_key in all_proj and ssp in all_proj[hdf_key]:
            comp_projections[ssp][label] = all_proj[hdf_key][ssp]

    # Total: sum MC samples across all available components
    n_samples = None
    total_samples = np.zeros((1, len(proj_years)))  # placeholder shape
    n_comps_summed = 0
    for hdf_key in COMP_LABELS:
        if hdf_key not in all_proj or ssp not in all_proj[hdf_key]:
            continue
        s = all_proj[hdf_key][ssp]['samples']
        if n_samples is None:
            n_samples = s.shape[0]
            total_samples = np.zeros((n_samples, len(proj_years)))
        total_samples += s
        n_comps_summed += 1

    comp_projections[ssp]['Total_sum'] = {
        'samples': total_samples,
        'median': np.median(total_samples, axis=0),
        'p5': np.percentile(total_samples, 5, axis=0),
        'p17': np.percentile(total_samples, 17, axis=0),
        'p83': np.percentile(total_samples, 83, axis=0),
        'p95': np.percentile(total_samples, 95, axis=0),
    }

print(f'Summed {n_comps_summed} components for each SSP')
print(f'Total samples shape: {total_samples.shape}')

## 3. Summary Table (2050, 2100, 2150)

In [ ]:
# ── Summary table: per-component + total at milestone years ──
milestone_years = [2050, 2100, 2150]
comp_order = ['Thermosteric', 'Glaciers', 'Greenland', 'EAIS', 'Peninsula', 'WAIS']

rows = []
for ssp in PROJ_SSPS:
    for yr in milestone_years:
        idx = np.argmin(np.abs(proj_years - yr))
        row = {'SSP': ssp, 'Year': yr}
        for cname in comp_order:
            if cname in comp_projections[ssp]:
                s = comp_projections[ssp][cname]['samples'][:, idx] * M_TO_MM
                row[cname] = f'{np.median(s):.0f} [{np.percentile(s, 5):.0f}, {np.percentile(s, 95):.0f}]'
            else:
                row[cname] = '—'
        s_tot = comp_projections[ssp]['Total_sum']['samples'][:, idx] * M_TO_MM
        row['Total'] = f'{np.median(s_tot):.0f} [{np.percentile(s_tot, 5):.0f}, {np.percentile(s_tot, 95):.0f}]'
        rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary = df_summary.set_index(['SSP', 'Year'])
display(df_summary)

## 4. Projection Fan Plots

In [ ]:
# ── Total GMSL projection: all SSPs ──
proj_mask = proj_years >= 2005
yr_plot = proj_years[proj_mask]

fig, ax = plt.subplots(figsize=(12, 7))
for ssp in PROJ_SSPS:
    p = comp_projections[ssp]['Total_sum']
    med = p['median'][proj_mask] * M_TO_MM
    lo5 = p['p5'][proj_mask] * M_TO_MM
    hi95 = p['p95'][proj_mask] * M_TO_MM
    lo17 = p['p17'][proj_mask] * M_TO_MM
    hi83 = p['p83'][proj_mask] * M_TO_MM
    color = SSP_COLORS.get(ssp, 'gray')
    ax.plot(yr_plot, med, color=color, lw=2, label=ssp)
    ax.fill_between(yr_plot, lo17, hi83, color=color, alpha=0.20)
    ax.fill_between(yr_plot, lo5, hi95, color=color, alpha=0.08)

ax.set_xlabel('Year')
ax.set_ylabel('GMSL (mm, rel. to 2005)')
ax.set_title('Total GMSL projection — component sum')
ax.set_xlim(2005, 2150)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_summation_total.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-component fans for SSP2-4.5 and SSP5-8.5 ──
fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

for ax, ssp in zip(axes, ['SSP2-4.5', 'SSP5-8.5']):
    for cname in comp_order:
        if cname not in comp_projections[ssp]:
            continue
        p = comp_projections[ssp][cname]
        med = p['median'][proj_mask] * M_TO_MM
        lo = p['p17'][proj_mask] * M_TO_MM
        hi = p['p83'][proj_mask] * M_TO_MM
        color = COMP_COLORS.get(cname, 'gray')
        ax.plot(yr_plot, med, color=color, lw=1.5, label=cname)
        ax.fill_between(yr_plot, lo, hi, color=color, alpha=0.15)

    ax.set_xlabel('Year')
    ax.set_title(f'Component projections — {ssp}')
    ax.set_xlim(2005, 2150)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.legend(fontsize=9, loc='upper left', ncol=2)

axes[0].set_ylabel('Sea-level contribution (mm)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_summation_fans.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Component Stack & Variance Decomposition

In [ ]:
# ── Stacked median contributions (SSP2-4.5) ──
ssp_show = 'SSP2-4.5'
fig, ax = plt.subplots(figsize=(12, 7))

bottoms = np.zeros(proj_mask.sum())
for cname in comp_order:
    if cname not in comp_projections[ssp_show]:
        continue
    med = comp_projections[ssp_show][cname]['median'][proj_mask] * M_TO_MM
    color = COMP_COLORS.get(cname, 'gray')
    ax.fill_between(yr_plot, bottoms, bottoms + med, color=color, alpha=0.7, label=cname)
    bottoms += med

# Overlay total median
tot_med = comp_projections[ssp_show]['Total_sum']['median'][proj_mask] * M_TO_MM
ax.plot(yr_plot, tot_med, 'k-', lw=2, label='Total (median)')

ax.set_xlabel('Year')
ax.set_ylabel('GMSL contribution (mm)')
ax.set_title(f'Stacked component medians — {ssp_show}')
ax.set_xlim(2005, 2150)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.legend(loc='upper left', fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_summation_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Variance decomposition ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, ssp in zip(axes, ['SSP2-4.5', 'SSP5-8.5']):
    comps_available = [c for c in comp_order if c in comp_projections[ssp]]
    fracs, raw_sum = compute_variance_fractions(
        ssp, comps_available, proj_years, comp_projections,
    )
    proj_mask_var = proj_years >= 2020
    yr_var = proj_years[proj_mask_var]

    bottoms = np.zeros(proj_mask_var.sum())
    for cname in comps_available:
        f = fracs[cname][proj_mask_var]
        color = COMP_COLORS.get(cname, 'gray')
        ax.fill_between(yr_var, bottoms, bottoms + f, color=color, alpha=0.7, label=cname)
        bottoms += f

    ax.set_xlim(2020, 2150)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Year')
    ax.set_title(f'Variance fraction — {ssp}')

axes[0].set_ylabel('Fraction of total variance')
axes[0].legend(fontsize=8, loc='upper right', ncol=2)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_summation_variance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. IPCC AR6 Comparison

In [ ]:
# ── Load IPCC AR6 total projections ──
ipcc_total = {}
for ssp, code in SSP_TO_CODE.items():
    data = read_ipcc_component_nc(CONF_BASE, 'medium_confidence', code, 'total')
    if data is not None:
        ipcc_total[ssp] = data
        print(f'  {ssp}: {data["years"][0]:.0f}–{data["years"][-1]:.0f}, '
              f'{len(data["quantiles"])} quantiles')
    else:
        print(f'  {ssp}: not found')

In [ ]:
# ── Our total vs IPCC total ──
fig, ax = plt.subplots(figsize=(12, 7))

for ssp in PROJ_SSPS:
    color = SSP_COLORS.get(ssp, 'gray')

    # Our projection
    p = comp_projections[ssp]['Total_sum']
    med = p['median'][proj_mask] * M_TO_MM
    lo = p['p5'][proj_mask] * M_TO_MM
    hi = p['p95'][proj_mask] * M_TO_MM
    ax.plot(yr_plot, med, color=color, lw=2, label=f'{ssp} (ours)')
    ax.fill_between(yr_plot, lo, hi, color=color, alpha=0.08)

    # IPCC
    if ssp in ipcc_total:
        ipcc_d = ipcc_total[ssp]
        ie = ipcc_extract(ipcc_d)
        ax.plot(ie['years'], ie['q50'], color=color, lw=1.5, ls='--', alpha=0.7,
                label=f'{ssp} (IPCC)')
        ax.fill_between(ie['years'], ie['q05'], ie['q95'],
                         color=color, alpha=0.05, linestyle='--')

ax.set_xlabel('Year')
ax.set_ylabel('GMSL (mm, rel. to ~2005)')
ax.set_title('Component sum vs IPCC AR6 total')
ax.set_xlim(2005, 2150)
ax.axhline(0, color='k', lw=0.5, ls='--')
ax.legend(fontsize=8, loc='upper left', ncol=2)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/component_summation_vs_ipcc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Numerical comparison at 2100 ──
print(f'{"SSP":<12} {"Ours (median [5,95])":<30} {"IPCC (median [5,95])":<30}')
print('-' * 72)
for ssp in PROJ_SSPS:
    idx_2100 = np.argmin(np.abs(proj_years - 2100))
    s = comp_projections[ssp]['Total_sum']['samples'][:, idx_2100] * M_TO_MM
    ours_str = f'{np.median(s):.0f} [{np.percentile(s, 5):.0f}, {np.percentile(s, 95):.0f}]'

    if ssp in ipcc_total:
        ipcc_stats = get_ipcc_stats(
            {ssp: {'total': ipcc_total[ssp]}}, ssp, 'total', year=2100,
        )
        if ipcc_stats is not None:
            ipcc_str = f'{ipcc_stats[1]:.0f} [{ipcc_stats[0]:.0f}, {ipcc_stats[2]:.0f}]'
        else:
            ipcc_str = '—'
    else:
        ipcc_str = '—'

    print(f'{ssp:<12} {ours_str:<30} {ipcc_str:<30}')